# Imports

In [ ]:
from itertools import product
import numpy as np
import pandas as pd
import tifffile
import cv2 as cv
from bioio import BioImage
import bioio_czi
from scipy.ndimage import distance_transform_cdt
from skimage.filters import threshold_otsu,threshold_multiotsu
from skimage.transform import resize

# Functions

In [397]:
def to_uint16(im):
    # make it uint16
    v_max = np.max(im)
    v_min = np.min(im)
    im = (65535*(im - v_min)/(v_max - v_min)).astype('uint16')    
    return im

In [398]:
def detect_top(file_path,
               blur_std=12,
               maxNeighbourhood=11,
               ksize_lap=11,
               ksize_var=21,
               max_kernel_size=50,
               thresh_blur_std=17,
               clip_data=False,
               thresh_const=0.08,
               returnLapVar=False,
               returnSurface=False,
               returnDist=True,
               surface_out=False,
               setAboveToZero=True,
               setAboveNegative=False,
               verbose=True):
    """
    Uses variance of Laplacian to detect the first point along z at which signal appears. 

    Parameters
    ----------
    file_path : str
        The path to the image that you are using to detect top.
    blur_std : int
        The image is blurred with this std (in pixels) so you want a size that decreases noise without blurring detail in image.
    maxNeighbourhood : int
        Whether to do a neighbourhood normalisation of the blurred data. If int then this is the kernel size.
    ksize_lap : int
        The kernel size of the initial Laplacian filter. I.e. about the size of the average edge.
    ksize_var : int
        The kernel size of the variance filter of the Laplacian. I.e. should capture several edges.
    max_kernel_size : int
        The kernel size of the neighbourhood maimum filter. I.e. should be bigger than areas that are in focus but don't have detail.
    thresh_blur_std : int
        The blur applied before the binarisation. I.e. size that you want the final surface to be smooth over.
    clip_data : False or int
        If > 0 then it clips to this percentile of the data (after blurring).
    thresh_meth : {'noise_factor','otsu'}
        How to threshold the variance of laplacian matrix. noise_factor sets threshold as a specfied multiple of the noise, which is defined as the mean of the first slice.
    surface_out : False or str
        Whether to save the top surface image and where to save it if so.          
    """

    returnList = []

    # load top signal data and smooth it a bit
    if file_path[-3:]=='czi':
        img_1 = BioImage(file_path, reader=bioio_czi.Reader)
        data_1 = img_1.get_image_data("ZYX",T=0,C=0)
    elif file_path[-3:]=='tif':
        data_1 = tifffile.imread(file_path)
        
    data_1_blur = np.array([cv.GaussianBlur(d,(0,0),blur_std) for d in data_1])

    del data_1
    
    if verbose:
        print('data loaded')

    if maxNeighbourhood:
        max_kernel = np.ones((maxNeighbourhood, maxNeighbourhood), np.uint8)
        data_1_blur_dil = np.array([cv.dilate(d,max_kernel) for d in data_1_blur.astype('float64')])
        data_1_blur = data_1_blur/data_1_blur_dil
        del data_1_blur_dil
    
    # calculate neighbourhood variance of the laplacian to measure 'focus'
    laplacian_1 = np.array([cv.Laplacian(d, cv.CV_64F, ksize=ksize_lap) for d in data_1_blur])
    del data_1_blur
    
    laplacian_1 = laplacian_1/np.max(laplacian_1)
    laplacian_1 = laplacian_1.astype('float32')
    mean_1 = np.array([cv.boxFilter(d, ddepth=-1, ksize=(ksize_var, ksize_var)) for d in laplacian_1]).astype('float32')
    meanSQ_1 = np.array([cv.boxFilter(d**2, ddepth=-1, ksize=(ksize_var, ksize_var)) for d in laplacian_1]).astype('float32')
    del laplacian_1
    var_1 = (meanSQ_1 - (mean_1**2)).astype('float16')
    del mean_1
    del meanSQ_1

    # do neighbourhood maximum (i.e. non-binary dilation) to enlarge areas of focus to account for regions without detail between focussesed regions with detail
    max_kernel = np.ones((max_kernel_size, max_kernel_size), np.uint8)
    var_1_dil = np.array([cv.dilate(d,max_kernel) for d in var_1.astype('float64')])

    del var_1

    if verbose:
        print('Laplacian variance calculated')

    var_1_dil_thresh = var_1_dil>=thresh_const

    if returnLapVar:
        returnList.append(var_1_dil)

    # take just the first pixel in z so that it is just top surface
    first_true_indices = np.argmax(var_1_dil_thresh, axis=0)    
    var_1_dil_thresh_first = np.zeros_like(var_1_dil_thresh, dtype=bool)
    Y, X = np.meshgrid(np.arange(var_1_dil_thresh.shape[1]), np.arange(var_1_dil_thresh.shape[2]), indexing='ij')
    var_1_dil_thresh_first[first_true_indices, Y, X] = var_1_dil_thresh.any(axis=0)

    del var_1_dil_thresh

    if verbose:
        print('binary surface calculated')
        
    if returnSurface:
        returnList.append(var_1_dil_thresh_first)
        
    if isinstance(surface_out,str):
        tifffile.imwrite(surface_out,var_1_dil_thresh_first)
        
    if returnDist:
        # calculate fast distance transform using chessboard thing (Euclidean is really slow)
        var_1_dil_thresh_first_dist = distance_transform_cdt(np.invert(var_1_dil_thresh_first), metric='chessboard').astype('int16')
    
        del var_1_dil_thresh_first
        
        # turn pixels above the top to zero
        first_zero_idx = np.argmax(var_1_dil_thresh_first_dist==0, axis=0) 
        Z, Y, X = var_1_dil_thresh_first_dist.shape
        z_indices = np.arange(Z)[:, None, None]  
        above_mask = z_indices < first_zero_idx
        if setAboveToZero:
            var_1_dil_thresh_first_dist[above_mask] = 0
        elif setAboveNegative:
            var_1_dil_thresh_first_dist[above_mask] = -var_1_dil_thresh_first_dist[above_mask]
    
        # also turn all z to zero for x-y positions that have no zeros... these are ones where no top surface was found
        mask = np.all(var_1_dil_thresh_first_dist!=0, axis=0)
        var_1_dil_thresh_first_dist[:, mask] = 32767
    
        if verbose:
            print('distance map calculated')
    
        returnList.append(var_1_dil_thresh_first_dist)

    if returnList:
        return returnList

In [ ]:
def extract_top_surface_data(
                            data_file_path,
                            surface_file_path,
                            out_csv,
                            z_step=0.1,
                            data_perc=99.9,
                            blur_std=7,
                            maxNeighbourhood=71,
                            ksize_lap=11,
                            ksize_var=31,
                            max_kernel_size=131,
                            thresh_blur_std=17,                                        
                            thresh_const=0.08,    
                            dist_map_1_out=False,
                            surface_out=False,
                            LapVar_out=False,
                            thresh_data_out=False,
                            verbose=False):
                                
    """
    This detects the top surface from one image and extracts signal from another image according to distance to that surface.
    
    Parameters
    ----------
    data_file_path : str
        The path to the image file of the signal you are pulling out.
    surface_file_path : str
        The path to the image file of the signal used to define the surface.
    out_csv : str
        Where to save the csv with all the data.
    surface_clip_perc : {int,float}
        The percentile you clip the surface data to, just to remove bright rubbish to improve otsu.
    dist_map_1_out : False or str
        Whether to save the distance map image and where to save it if so.
    surface_out : False or str
        Whether to save the top surface image and where to save it if so.        
    LapVar_out : False or str
        Whether to save the Laplacian variance (blurred and dilated) image and where to save it if so.
    """
    if data_file_path[-3:]=='czi': 
        img_1 = BioImage(data_file_path, reader=bioio_czi.Reader)
        data_1 = img_1.get_image_data("ZYX",T=0,C=0)
    elif data_file_path[-3:]=='tif':
        data_1 = tifffile.imread(data_file_path)
    data_1_blur = np.array([cv.GaussianBlur(d,(0,0),blur_std) for d in data_1])
    
    NZ_1 = data_1.shape[0]
    
    del data_1
    
    perc_1 = np.percentile(data_1_blur,data_perc)
    data_1_blur_thresh = data_1_blur>perc_1
    
    del data_1_blur

    if isinstance(thresh_data_out,str):
        tifffile.imwrite(thresh_data_out,data_1_blur_thresh)        
    
    LapVar_dist_data_1 = []
    pix_dist_data_1 = []
    Npix_atDist_1 = []
    
    (LapVar_1,dist_map_1) = detect_top(surface_file_path,
                                        blur_std=blur_std,
                                        maxNeighbourhood=maxNeighbourhood,
                                        ksize_lap=ksize_lap,
                                        ksize_var=ksize_var,
                                        max_kernel_size=max_kernel_size,
                                        thresh_blur_std=thresh_blur_std,                                        
                                        thresh_const=thresh_const,
                                        returnSurface=False,
                                        surface_out=surface_out,
                                        returnDist=True,
                                        setAboveNegative=True,
                                        setAboveToZero=False,
                                        returnLapVar=True,
                                        verbose=verbose)
    
    if isinstance(dist_map_1_out,str):
        tifffile.imwrite(dist_map_1_out,dist_map_1)

    if isinstance(LapVar_out,str):
        tifffile.imwrite(LapVar_out,LapVar_1)
    
    minDist_1 = dist_map_1.min()
    maxDist_1 = dist_map_1[dist_map_1!=32767].max()
    
    for d in range(minDist_1,maxDist_1+1):
        if verbose:
            if d%20==0:
                print(d)
        dist_mask = dist_map_1==d
        Npix_atDist_1.append(np.sum(dist_mask))
        LapVar_dist_data_1.append(np.mean(LapVar_1[dist_mask]))

    del LapVar_1
    
    dist_map_2 = resize(dist_map_1, 
                        data_1_blur_thresh.shape, 
                        order=1, 
                        mode='reflect', 
                        anti_aliasing=True, 
                        preserve_range=True)

    del dist_map_1     
    dist_map_2 = dist_map_2.astype('uint16')   
    
    for d in range(minDist_1,maxDist_1+1):
        if verbose:
            if d%20==0:
                print(d)
        dist_mask2 = dist_map_2==d
        pix_dist_data_1.append(np.sum(data_1_blur_thresh[dist_mask2]))

    dist_1_um = [z_step*(i) for i in range(minDist_1,maxDist_1+1)]
    
    # this normalisation is in case stiff and soft have slightly different numbers of pixels in the top 0.1 percentile
    pix_dist_data_1_norm = pix_dist_data_1/np.sum(pix_dist_data_1)
    
    # this normalisation is needed because each distance will have different numbers of pixels, i.e. it makes fraction of positive pixels rather than number
    pix_dist_data_1_norm_npixNorm = np.array(pix_dist_data_1_norm)/np.array(Npix_atDist_1)
    
    LapVar_dist_data_1_norm = LapVar_dist_data_1/np.max(LapVar_dist_data_1)
    
    df_data_1 = {'distance (um)':dist_1_um,
                     'no. of pixels at distance':Npix_atDist_1,
                     'LapVar signal':LapVar_dist_data_1,
                     'number of signal pixels':pix_dist_data_1,
                     'LapVar signal normalised':LapVar_dist_data_1_norm,
                     'number of signal pixels normalised':pix_dist_data_1_norm,
                     'fraction (at dist) of signal pixels normalised':pix_dist_data_1_norm_npixNorm}
    
    df_1 = pd.DataFrame(df_data_1)
    
    df_1.to_csv(out_csv)

# Parameters to set

In [ ]:
# paths to the files with the signal you are measuring 
fp_stiff_data = r''
fp_soft_data =r''

# paths to the files of the HCS image data which the code uses to detect the top surface of the cells
# this should be the files BEFORE SIM processing (because SIM processing gives artefacts that look like in-focus signal)
# the dimensions don't need to be the same as the signal data files above, the code will account for differences
# (SIM processing increases the size of x,y,z dimensions, you can use SIM for signal and non-SIM for HCS)
# BUT you DO need to remove the 'SIM dimension' from the non-SIM-processed data
# i.e. in raw SIM files there are many images of the same z-slice (=the SIM-dimension), which disappear when you do the SIM processing... just save a new image with just the first of these images
fp_stiff_HCS_noSim = r''
fp_soft_HCS_noSim =  r''

# the file path of the csv that will be saved containing extracted data
out_csv_stiff =r''
out_csv_soft = r''

# where to save the Laplacian variance, distance map and top surface as a tiff
# just do this if you are trying to figure out bugs
LapVar_out_soft = False 
dist_map_1_out_soft = False 
surface_out_soft = False 
LapVar_out_stiff = False 
dist_map_1_out_stiff = False 
surface_out_stiff = False 
thresh_data_out_stiff = False
thresh_data_out_soft = False


# the size in um of one z-slice in the HCS data that you provide
z_step=0.3

# other parameters
data_perc=99.9
blur_std = 7
maxNeighbourhood = 71
ksize_lap = 11
ksize_var=31
max_kernel_size=131
thresh_blur_std=17

# Run function

In [469]:
%%time

extract_top_surface_data(fp_stiff_data,
                           fp_stiff_HCS_noSim,
                           out_csv_stiff,
                           z_step=z_step,
                           data_perc=data_perc,
                           blur_std=blur_std,
                           maxNeighbourhood=maxNeighbourhood,
                           ksize_lap=ksize_lap,
                           ksize_var=ksize_var,
                           max_kernel_size=max_kernel_size,
                           thresh_blur_std=thresh_blur_std,                             
                           thresh_const=0.08,
                           dist_map_1_out=dist_map_1_out_stiff,
                           surface_out=surface_out_stiff,
                           LapVar_out=LapVar_out_stiff,
                           thresh_data_out=thresh_data_out_stiff,
                           verbose=True)

data loaded
Laplacian variance calculated
binary surface calculated
distance map calculated
0
20
40
60
80
0
20
40
60
80
CPU times: total: 1min 21s
Wall time: 50.4 s


In [ ]:
%%time

extract_top_surface_data(fp_soft_data,
                         fp_soft_HCS_noSim,
                         out_csv_soft,
                         z_step=z_step,
                         data_perc=data_perc,
                         blur_std=blur_std,
                         maxNeighbourhood=maxNeighbourhood,
                         ksize_lap=ksize_lap,
                         ksize_var=ksize_var,
                         max_kernel_size=max_kernel_size,
                         thresh_blur_std=thresh_blur_std,                                        
                         thresh_const=0.08,    
                         dist_map_1_out=dist_map_1_out_soft,
                         surface_out=surface_out_soft,
                         LapVar_out=LapVar_out_soft,
                         thresh_data_out=thresh_data_out_soft,
                         verbose=True)

data loaded
Laplacian variance calculated
binary surface calculated
distance map calculated
-20
0
20
40
60
80
